# Implementing Bag of Words with N-grams
With CountVectorizer, Bag of Words plus n-grams is implemented by setting the ngram_range parameter. It will build a vocabulary of contiguous word sequences and create a document-by-feature count matrix, just like normal BoW—except features can be unigrams, bigrams, trigrams, or a mix.

## ngram_range values
ngram_range=(1, 1)	-  Only unigrams: machine, learning

ngram_range=(2, 2)	-  Only bigrams: machine learning, natural language

ngram_range=(3, 3)	-  Only trigrams: natural language processing

ngram_range=(1, 2)	- Unigrams + bigrams

ngram_range=(1, 3)	- Unigrams + bigrams + trigrams

ngram_range=(2, 3)	- Bigrams + trigrams, no unigrams

In [1]:
import re
import nltk
from nltk.tokenize import word_tokenize, sent_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.corpus import wordnet
from sklearn.feature_extraction.text import CountVectorizer

In [2]:
import pandas
messages = pandas.read_csv('data/sms_spam_collection/sms_collection', sep='\t',names=['label', 'message'])

In [3]:
messages.head()

,label,message
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


In [4]:
lemmatizer = WordNetLemmatizer()

In [5]:
def get_wordnet_pos(tag):
    if tag.startswith('J'):
        return wordnet.ADJ
    elif tag.startswith('V'):
        return wordnet.VERB
    elif tag.startswith('N'):
        return wordnet.NOUN
    elif tag.startswith('R'):
        return wordnet.ADV
    else:
        return wordnet.NOUN

In [6]:
# go thru each message -> remove non-alphabetic characters -> lowercase -> tokenize -> remove stopwords -> lemmatize -> join back to string
corpus = []
for i in range(len(messages)):
    message = re.sub('[^a-zA-Z]', ' ', messages['message'][i])
    message = message.lower()
    message = word_tokenize(message)
    tagged = nltk.pos_tag(message)
    message = [lemmatizer.lemmatize(word, pos=get_wordnet_pos(tag)) for word, tag in tagged if word not in set(stopwords.words('english'))]
    corpus.append(' '.join(message))

In [7]:
corpus[:5]

['go jurong point crazy available bugis n great world la e buffet cine get amore wat',
 'ok lar joking wif u oni',
 'free entry wkly comp win fa cup final tkts st may text fa receive entry question std txt rate c apply',
 'u dun say early hor u c already say',
 'nah think go usf live around though']

In [8]:
count_vectorizer = CountVectorizer(max_features=2500, ngram_range=(1, 2))
# here max_features=2500 means we will only keep the 2500 most frequent words in our vocabulary. 

ngram_range=(1, 2) means: include all unigrams and bigrams

In [9]:
X_bow_ngrams = count_vectorizer.fit_transform(corpus).toarray()

In [10]:
print(X_bow_ngrams.shape)  # (5572, 2500) - 5572 messages and 2500 features (words or n-grams)

(5572, 2500)


In [11]:
print(count_vectorizer.get_feature_names_out()[:200])

['aathi' 'abi' 'abiola' 'able' 'abt' 'ac' 'acc' 'accept' 'access'
 'accidentally' 'account' 'account statement' 'ache' 'across' 'across sea'
 'act' 'action' 'activate' 'actually' 'ad' 'add' 'address' 'admirer'
 'admirer look' 'adult' 'advance' 'advice' 'affair' 'affection' 'aft'
 'afternoon' 'afternoon love' 'aftr' 'age' 'ago' 'ah' 'aha' 'ahead'
 'ahmad' 'aight' 'aint' 'air' 'airport' 'aiyah' 'aiyo' 'al' 'alert' 'alex'
 'allah' 'allow' 'almost' 'alone' 'already' 'alright' 'alrite' 'also'
 'always' 'amazing' 'amount' 'amp' 'amp say' 'amt' 'an' 'angry'
 'announcement' 'another' 'another day' 'answer' 'anybody' 'anymore'
 'anyone' 'anything' 'anything lor' 'anytime' 'anytime network' 'anyway'
 'anyway go' 'anyways' 'apartment' 'apartment go' 'app' 'apparently'
 'apply' 'appointment' 'appreciate' 'appt' 'april' 'ar' 'arcade' 'ard'
 'ard smth' 'area' 'argue' 'argument' 'arm' 'armand' 'around' 'around lt'
 'arrange' 'arrest' 'arrive' 'arrive please' 'art' 'as' 'asap' 'asap box'
 'ask' 'ask c

We can see some bigram Phrases such as 'award call', 'award either', 'award prize', 'award sipix', 'bonus caller'

In [12]:
count_vectorizer_trigram_only = CountVectorizer(max_features=2500, ngram_range=(3, 3))

In [13]:
X_bow_trigrams = count_vectorizer_trigram_only.fit_transform(corpus).toarray()

In [14]:
print(count_vectorizer_trigram_only.get_feature_names_out()[:20])

['aathi love lot' 'accept brother sister' 'accept day accept'
 'account lt gt' 'account statement show' 'admirer look make'
 'advise follow recent' 'affection care luv' 'afternoon love go'
 'afternoon wife call' 'alfie moon child' 'amp eve go' 'amp fill gap'
 'amp miracle tomorrow' 'amp need clean' 'amp say always' 'amp say wait'
 'amp successful day' 'amp sweet dream' 'amp take care']


These are the trigram only phrases

In [15]:
bow_df = pandas.DataFrame(
    X_bow_trigrams,
    columns=count_vectorizer_trigram_only.get_feature_names_out(),
    index=[f"doc_{i + 1}" for i in range(len(corpus))]
)

However, feature count grows very quickly as you add larger n-grams, so (1, 2) is usually the first setting worth testing for a traditional text-classification baseline. N-gram feature spaces grow rapidly as n increases

## Useful parameters with n-gram BoW

In [19]:
vectorizer = CountVectorizer(
    ngram_range=(1, 2),
    lowercase=True,
    stop_words="english",
    min_df=2,
    max_df=0.90,
    max_features=2_000
)

| Parameter            | What it does                                       | Why it helps with n-grams                                               |
| -------------------- | -------------------------------------------------- | ----------------------------------------------------------------------- |
| ngram_range=(1, 2)   | Uses unigrams and bigrams                          | Captures words plus short phrases                                       |
| lowercase=True       | Lowercases text before tokenization                | Avoids separate features for NLP and nlp                                |
| stop_words="english" | Removes scikit-learn’s built-in English stop words | Can reduce common-word features, though evaluate carefully for negation |
| min_df=2             | Retains features found in at least 2 documents     | Removes one-off, noisy phrases                                          |
| max_df=0.90          | Removes terms in more than 90% of documents        | Removes corpus-specific very common terms                               |
| max_features=2_000  | Keeps only the most frequent features              | Controls memory and dimensionality                                      |

CountVectorizer documents ngram_range as the range of word or character n-grams, while options such as max_df, min_df, and max_features help control the learned vocabulary.

In [20]:
X = vectorizer.fit_transform(corpus)

In [21]:
X.shape

(5572, 2000)

## Important distinction
Using n-grams with CountVectorizer is still Bag of Words.

For example:

CountVectorizer(ngram_range=(1, 2))

creates a bag of unigrams and bigrams. It counts features like "not good", but it does not preserve the full sentence sequence or long-distance structure. "not good" is recognized as one local phrase feature; the model still does not understand grammar or meaning in the way a contextual model does.

## Word n-grams vs character n-grams
By default, CountVectorizer uses word n-grams because analyzer="word" is the default.

In [22]:
word_vectorizer = CountVectorizer(
    ngram_range=(1, 2),
    analyzer='word')

To generate character n-grams instead:

In [23]:
char_vectorizer = CountVectorizer(
    ngram_range=(2, 5),
    analyzer='char')

Character n-grams are useful when spelling variation, typos, morphology, or out-of-vocabulary words matter; word n-grams are more interpretable for phrase-based features.